### Enformer sequence generation:
- 270bp sequences
- get sequence context of genome file around these sequences 500bp upstream and 500bp downstream
- for all variants and control variants
- Quick way: give Mohan the fasta like tsv of the design file and let him generate the predictions of enformer for each 270bp sequence

In [7]:
# connection test:
# ! head /data/cephfs-1/work/projects/cubit/18.12/static_data/reference/GRCh38/hs38/hs38.fa

In [8]:
# helpful functions: 
import hashlib 

def shorten_GC_Kircher_header(header, sequence):
    if hf.get_label(header) == 'GC_Kircher':
        if "REF_" in header:
            return 'GC_Kircher:REF_oligo_' + hashlib.md5(sequence.encode()).hexdigest()
        elif "ALT_" in header:
            return 'GC_Kircher:ALT_oligo_' + hashlib.md5(sequence.encode()).hexdigest()
    else:
        return header

def control_ID_postprocessing(ID):
    """
    Manupulate the IDs of the controls because of changes in headers downstream
    replace "," with "~"
    and for "GC_Mendelian_variants": replace ">" with "*"
    """
    if hf.get_label(ID) == 'GC_Mendelian_variants':
        ID = ID.replace(">", "*")
    if hf.get_label(ID) != 'cardiac_neuro_cava_random':
        ID = ID.replace(",", "~")
    return ID

def control_ID_processing_variant_mapping(ID):
    """
    Process headers in a way they occur in other files like variant_region_map
    """
    if hf.get_label(ID) == 'GC_Mendelian_variants':
        ID = ID.replace("*", ">")
    ID = ID.replace("~", ",")
    return ID

def get_chrom_pos_ref_alt(ID): 
    """Returns the last part of the ID column (after the last '|')"""
    if hf.get_label(ID) == 'cardiac_neuro_cava_random':
        return ID.split('|')[-1]
    else:
        return unknown

In [35]:
import pandas as pd 
import yaml 

# import local module helpful functions as hf (/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/00_helpful_functions/helpful_functions.py)
import sys
sys.path.append("/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/00_helpful_functions")
import helpful_functions as hf

# # reload helpful_functions module
# import importlib
# importlib.reload(hf)


config_path = "/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/global80K_config.yaml"
config_path = "../../global80K_config.yaml"
# config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
# load config file
with open(config_path, "r") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)


### Quick way: fasta-like tsv for all sequences in the design

In [36]:
designed_sequences = hf.fasta_to_dataframe(config['files']['final_design']['design_fasta'])
designed_sequences

# shorten headers of GC_Kircher
designed_sequences['short_header'] = designed_sequences.apply(lambda x: shorten_GC_Kircher_header(x['header'], x['sequence']), axis=1)
designed_sequences['sequence_without_adapter'] = designed_sequences['sequence'].str.slice(15, 285)

designed_sequences = designed_sequences[['short_header', 'sequence_without_adapter']]
# rename columns to header, sequence
designed_sequences = designed_sequences[['short_header', 'sequence_without_adapter']].rename(columns={'short_header': 'header', 'sequence_without_adapter': 'sequence'})
# store tsv file
designed_sequences.to_csv(config['files']['creating']['enformer_prediction_tsv'], sep="\t", index=False)

### For getting the context of 1kb around the sequences get a region file (bed like and create fasta file from there)
- find positions of all references and elements/regions
- for alternative sequences get the regions from their reference and add the variant sequence inbetween
- way more work for controls, because matching of headers is not working without problems

In [10]:
# get the seqeuence headers you would like to get predictions for
variant_table_path = config['files']['final_design']['variant_table'] # variant_region_file with ',': 47044 variants
# includes duplicates for cardiac_neuro_cava_random but the deduplicated does not contain the variant controls
variant_table_df = pd.read_csv(variant_table_path, sep="\t")
variant_table_df
variant_table_path_deduplicated = config['files']['final_design']['variant_table_deduplicated'] # variant_region_file with ',': 47044 variants
deduplicated_variant_table_df = pd.read_csv(variant_table_path_deduplicated, sep="\t")
deduplicated_variant_table_df # 46374 => only cardiac_neuro_cava_random
# only control information:
c_variant_table_df = pd.read_csv(config['files']['final_design']['variant_region_map_controls'], sep="\t")
c_variant_table_df

,Variant,Region,REF_ID,ALT_ID
0,GC_Atrial_fib:rs74541936,GC_Atrial_fib:rs74541936|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs74541936|KCNN3|STARR-seq-A...
1,GC_Atrial_fib:rs34292822,GC_Atrial_fib:rs34292822|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs34292822|KCNN3|STARR-seq-A...
2,GC_Atrial_fib:rs12754189,GC_Atrial_fib:rs12754189|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs12754189|KCNN3|STARR-seq-A...
3,GC_Atrial_fib:rs36088503,GC_Atrial_fib:rs36088503|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs36088503|KCNN3|STARR-seq-A...
4,GC_Atrial_fib:rs76749863,"GC_Atrial_fib:rs1218584|KCNN3|STARR-seq-AF,rs7...",GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF...
...,...,...,...,...
665,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618
666,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757
667,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373
668,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656


In [11]:
# check if MK is in the variant map table => no it is not
c_variant_table_df['Variant'].str.split(':').str[0].value_counts()
# Variant
# GC_Selvarajan            198 REF
# GC_Kircher               198 REF
# GC_Mendelian_variants    174 REF 
# C_positive_heart_CAD      49 REF
# GC_Atrial_fib             23 REF
# GC_Mohlke                 20 REF
# GC_Liang                   8 REF

Variant
GC_Selvarajan            198
GC_Kircher               198
GC_Mendelian_variants    174
C_positive_heart_CAD      49
GC_Atrial_fib             23
GC_Mohlke                 20
GC_Liang                   8
Name: count, dtype: int64

#### Add chrom pos ref alt to the sequences
- first for cardiac_neuro_cava_random (majority)
- controls: maybe manually: add region information /data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/final_design/regions.bed.gz

In [12]:
# # open region file: only controls? no all regions
# region_file_path = config['files']['final_design']['region_bed']
# region_file = pd.read_csv(region_file_path, sep="\t", header=None)
# region_file.columns = ['chr', 'start', 'end', 'ID', 'score', 'strand']
# region_file
# # add label column
# region_file['label'] = region_file['ID'].apply(hf.get_label)
# # region_file['label'].value_counts()



In [13]:
deduplicated_variant_table_df.head()

,Variant,Region,REF_ID,ALT_ID
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...


In [14]:
deduplicated_variant_table_df['chrom_pos_ref_alt'] = deduplicated_variant_table_df['Variant'].apply(get_chrom_pos_ref_alt)
deduplicated_variant_table_df

,Variant,Region,REF_ID,ALT_ID,chrom_pos_ref_alt
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,1-2179591-T-C
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,1-2191444-G-A
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,1-2192015-G-T
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,1-2192366-T-G
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...,1-2193142-G-A
...,...,...,...,...,...
46369,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,X-154545206-A-G
46370,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,X-154549923-T-G
46371,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,X-154552289-C-T
46372,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:G6PD|ENSG00000160211...,cardiac_neuro_cava_random:REF_G6PD|ENSG0000016...,cardiac_neuro_cava_random:ALT_G6PD|ENSG0000016...,X-154552371-G-A


In [15]:
# add here the region information to the cardiac_neuro_cava_random group (don't filter for ALT_ REF_ ) 
# bed file has ids with ","

#### Do the same for the controls: 

##### Use bed and vcf file (not working because for 2 groups there is no region information provided)
- solution: use blat again for the controls variant sequences

In [16]:
# # add chrom pos ref alt and strand to c_variant_table_df and deduplicated_variant_table_df by left joining preprocessed region_file
# region_file = region_file[['ID', 'chr', 'start', 'end', 'strand']]
# region_control_variant_df = c_variant_table_df.merge(region_file, left_on='Region', right_on='ID', how='left')
# region_control_variant_df
# # not mergable: 
# not_mergable = region_control_variant_df[region_control_variant_df['ID'].isna()]
# not_mergable['label'] = not_mergable['Region'].apply(hf.get_label)
# # not_mergable['label'].value_counts()
# # # GC_Mendelian_variants    174
# # # C_positive_heart_CAD      49
# region_file['label'] = region_file['ID'].apply(hf.get_label)
# region_file['label'].value_counts()

In [17]:
# # check if I can find all REF_ID and ALT_ID for the controls in the design file as header

# # first replace "," with "~"

# # # replace "," with "~" for the controls because current design file has only ~
# # rename REF_ID to REF_ID_with_tilde and ALT_ID to ALT_ID_with_tilde
# c_variant_table_df.rename(columns={"REF_ID": "REF_ID_with_comma", "ALT_ID": "ALT_ID_with_comma"}, inplace=True)
# c_variant_table_df['REF_ID'] = c_variant_table_df['REF_ID_with_comma'].apply(control_ID_postprocessing)
# c_variant_table_df['ALT_ID'] = c_variant_table_df['ALT_ID_with_comma'].apply(control_ID_postprocessing)
# # remove REF_ID_with_comma and ALT_ID_with_comma
# c_variant_table_df.drop(columns=["REF_ID_with_comma", "ALT_ID_with_comma"], inplace=True)
# c_variant_table_df

# c_variant_table_df
# quick solution: take all control sequences from design and blat them
# need step inbetween some headers are too long for blat (GC_Kircher)

# find all reference sequences among the control sequences


##### Use blat

In [18]:
def header_from_ref(header):
    """Will return true if the header is a reference header"""
    header_part = header.split(":")[1]
    if header_part.startswith("REF_"):
        return True
    return False

In [19]:
# load design file
designd_sequences = hf.fasta_to_dataframe(config['files']['final_design']['design_fasta'])

designd_sequences_controls = designd_sequences[designd_sequences['header'].apply(hf.is_control)]
designd_sequences_controls # 6275 control sequences
designd_sequences_controls['sequence_without_adapter'] = designd_sequences_controls['sequence'].str.slice(15, 285)

# shorten headers of GC_Kircher
designd_sequences_controls['short_header'] = designd_sequences_controls.apply(lambda x: shorten_GC_Kircher_header(x['header'], x['sequence']), axis=1)

# find all references of the designed sequences (found that all sequences which are variant controls have REF_ in headers which are used as references)
designd_sequences_controls = designd_sequences_controls[designd_sequences_controls['short_header'].apply(header_from_ref)]
designd_sequences_controls['sequence_length'] = designd_sequences_controls['sequence'].apply(len)

print(designd_sequences_controls.shape[0]) # 312 headers from the control sequences are references
pd.DataFrame(designd_sequences_controls['short_header'].str.split(':').str[0].value_counts().reset_index())
# Numbers of references per control group:
# GC_Selvarajan	            165
# GC_Mendelian_variants	    48
# C_positive_heart_CAD	    48
# GC_Atrial_fib	            21
# GC_Mohlke	                17
# GC_Liang	                8
# GC_Kircher	            5


# # write to fasta file
# output_path = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/results/control_sequences/control_variant_reference_sequence.fasta'

# with open(output_path, 'w') as f:
#     for index, row in designd_sequences_controls.iterrows():
#         f.write(f">{row['short_header']}\n{row['sequence_without_adapter']}\n")



312


/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/1382868222.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  designd_sequences_controls['sequence_without_adapter'] = designd_sequences_controls['sequence'].str.slice(15, 285)
/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1774004/1382868222.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  designd_sequences_controls['short_header'] = designd_sequences_controls.apply(lambda x: shorten_GC_Kircher_header(x['header'], x['seque

,short_header,count
0,GC_Selvarajan,165
1,GC_Mendelian_variants,48
2,C_positive_heart_CAD,48
3,GC_Atrial_fib,21
4,GC_Mohlke,17
5,GC_Liang,8
6,GC_Kircher,5


##### Check blat results 
- perfect matches found on hg38: 309
- input 312 sequences => 3 reference sequences not found in the genome
- which sequences? (both have 1 gap according to blat)
    - GC_Mendelian_variants:REF_chr8:11703860G*T|GATA4
    - GC_Mendelian_variants:REF_chr8:11703890AG*A|GATA4
    - GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3
- all 4 example sequences could be found in ucsc
- Next: generate bed formated file:
  - add the region to the associated alternative sequence as well (for sequence context up to 1kb)
  - problem: not all found reference sequences of candidate variant controls are in the variant control map
  - how many of the REF_ID and ALT_ID entries of the variant control map are in the final design?
    - problem with "," and "~"?
    - problem with the generation of the variant control map?
    - which sequences are not matchable?
      - example not matchable because occures mutliple times in REF_ID (`GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3`) (original header from the design file can not be matched with REF_ID)
      - Not in variant region map (probably just element sequences)
        - GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd_tile1-1
        - GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd_tile1-1
      - have duplicated header tag and cannot be matched uniquely
        - GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4_headerDuplicate2_2_headerDuplicate1_2
        - GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4_headerDuplicate2_2_headerDuplicate1_2


In [20]:
def set_modified_chromosome(row):
    """i.e. from NC_000001.11 to chr1, ..."""
    if '23' in row['T_name']:
        return 'chrX'
    if '24' in row['T_name']:
        return 'chrY'
    chr_number = int(row['T_name'].split('_')[1].split('.')[0])
    return 'chr%s'%(chr_number) 

In [21]:
# create tsv from blat results
! tail -n +6 /data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_variant_control_sequences_matched_file.psl > /data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_variant_control_sequences_matched_file.tsv

In [22]:
# load blat results and check if only one perfect match of each sequence

header_names = ['match', 'mismatch', 'rep_match', 'Ns', 'Q_gap_count', 'Q_gap_bases', 'T_gap_count', 'T_gap_bases', 'strand', 'Q_name', 'Q_size', 'Q_start', 'Q_end', 'T_name', 'T_size', 'T_start', 'T_end', 'block_count', 'blockSizes', 'qStarts', 'tStarts']
path_to_blat_results = '/data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_variant_control_sequences_matched_file.tsv'
blat_results = pd.read_csv(path_to_blat_results, sep="\t", header=None)
blat_results.columns = header_names
blat_results

# filter: number of matches=270, 
perfect_matches = blat_results[blat_results['match'] == 270]
perfect_matches.shape # 314
# filter for blockSizes = 270
perfect_matches = perfect_matches[perfect_matches['blockSizes'] == '270,']
perfect_matches.shape # 311
# check all columns for unique values and did not find any suspecious values
perfect_matches.T_name.value_counts() # 0 
# filter for matches startwith "NC_" in T_name
perfect_matches = perfect_matches[perfect_matches['T_name'].str.startswith('NC_')]
perfect_matches.shape # 309
perfect_matches.Q_name.nunique() # 309

# get subset of interesting columns
interesting_blat_results = ['Q_name', 'match', 'strand', 'T_name', 'T_start', 'T_end']
interesting_blat_results

perfect_matches = perfect_matches[interesting_blat_results]
perfect_matches



,Q_name,match,strand,T_name,T_start,T_end
0,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154813248,154813518
1,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154839744,154840014
2,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154840018,154840288
3,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154840331,154840601
4,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,270,+,NC_000001.11,154860467,154860737
...,...,...,...,...,...,...
5563,C_positive_heart_CAD:REF_rs7865618,270,+,NC_000009.12,22030870,22031140
5939,C_positive_heart_CAD:REF_rs4977757,270,+,NC_000009.12,22094195,22094465
6625,C_positive_heart_CAD:REF_rs1537373,270,+,NC_000009.12,22103206,22103476
6626,C_positive_heart_CAD:REF_rs10811656,270,+,NC_000009.12,22124337,22124607


In [23]:
# merge the region results to the reference sequences
mapped_blat_results = designd_sequences_controls.merge(perfect_matches, left_on='short_header', right_on='Q_name', how='left')

# mapped_blat_results[mapped_blat_results['Q_name'].isna()] # 3 results
mapped_blat_results = mapped_blat_results[~mapped_blat_results['Q_name'].isna()]

mapped_blat_results['T_start'] = mapped_blat_results['T_start'].astype(int)
mapped_blat_results['T_end'] = mapped_blat_results['T_end'].astype(int)


# add chrom chromStart chromEnd
mapped_blat_results['chrom'] = mapped_blat_results.apply(set_modified_chromosome, axis=1)
mapped_blat_results

mapped_blat_results['chromStart'] = mapped_blat_results['T_start']
mapped_blat_results['chromEnd'] = mapped_blat_results['T_end']
mapped_blat_results

# drop inplace match, T_name, T_start, T_end
mapped_blat_results.drop(columns=['match', 'T_name', 'T_start', 'T_end'], inplace=True)
mapped_blat_results
# check with ucsc:
mapped_blat_results['header'].str.split(':').str[0].value_counts()
# # GC_Selvarajan (found it in ucsc) https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A3036131%2D3036161&hgsid=2101608398_MTV7V94UoV8fSg0LQWbEvX4v0gqy
# mapped_blat_results[mapped_blat_results['header'].str.split(':').str[0] == 'GC_Selvarajan']['short_header'].to_list()
# mapped_blat_results[mapped_blat_results['header'] == 'GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1']
# print(mapped_blat_results[mapped_blat_results['header'] == 'GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1']['chrom'])
# print(mapped_blat_results[mapped_blat_results['header'] == 'GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1']['chromStart'])
# print(mapped_blat_results[mapped_blat_results['header'] == 'GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1']['chromEnd'])
# print(mapped_blat_results[mapped_blat_results['header'] == 'GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1']['strand'])

# print(mapped_blat_results[mapped_blat_results['header'] == 'GC_Selvarajan:REF_rs9660819|STARR-seq-HepG2~rs9661525|STARR-seq-HepG2_fwd_tile1-1']['sequence_without_adapter'])


# # GC_Liang found in ucsc (https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr2%3A208556887%2D208556907&hgsid=2101608398_MTV7V94UoV8fSg0LQWbEvX4v0gqy)
# mapped_blat_results['header'].str.split(':').str[0].value_counts()
# mapped_blat_results[mapped_blat_results['header'].str.split(':').str[0] == 'GC_Liang']['short_header'].to_list()
# header_of_interest = 'GC_Liang:REF_rs1036014|Liang_fwd_tile1-1'
# mapped_blat_results[mapped_blat_results['header'] == header_of_interest]
# print(mapped_blat_results[mapped_blat_results['header'] == header_of_interest]['chrom'])
# print(mapped_blat_results[mapped_blat_results['header'] == header_of_interest]['chromStart'])
# print(mapped_blat_results[mapped_blat_results['header'] == header_of_interest]['chromEnd'])
# print(mapped_blat_results[mapped_blat_results['header'] == header_of_interest]['strand'])
# print(mapped_blat_results[mapped_blat_results['header'] == header_of_interest]['sequence_without_adapter'])


# # GC_Mendelian_variants: found in ucsc: https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A23219240%2D23219270&hgsid=2101608398_MTV7V94UoV8fSg0LQWbEvX4v0gqy
# mapped_blat_results['header'].str.split(':').str[0].value_counts()
# mapped_blat_results[mapped_blat_results['header'].str.split(':').str[0] == 'GC_Mendelian_variants']['short_header'].to_list()
# header_of_interest = 'GC_Mendelian_variants:REF_chr10:23219376A*C|PTF1A'
# mapped_blat_results[mapped_blat_results['header'] == header_of_interest]
# print(mapped_blat_results[mapped_blat_results['header'] == header_of_interest]['chrom'])
# print(mapped_blat_results[mapped_blat_results['header'] == header_of_interest]['chromStart'])
# print(mapped_blat_results[mapped_blat_results['header'] == header_of_interest]['chromEnd'])
# print(mapped_blat_results[mapped_blat_results['header'] == header_of_interest]['strand'])
# print(mapped_blat_results[mapped_blat_results['header'] == header_of_interest]['sequence_without_adapter'])


# # GC_Kircher
# mapped_blat_results['header'].str.split(':').str[0].value_counts()
# mapped_blat_results[mapped_blat_results['header'].str.split(':').str[0] == 'GC_Kircher']['short_header'].to_list()
# header_of_interest = 'GC_Kircher:REF_oligo_c90ef99e4413a4641966516092135115'
# mapped_blat_results[mapped_blat_results['short_header'] == header_of_interest]
# print(mapped_blat_results[mapped_blat_results['short_header'] == header_of_interest]['chrom'])
# print(mapped_blat_results[mapped_blat_results['short_header'] == header_of_interest]['chromStart'])
# print(mapped_blat_results[mapped_blat_results['short_header'] == header_of_interest]['chromEnd'])
# print(mapped_blat_results[mapped_blat_results['short_header'] == header_of_interest]['strand'])
# print(mapped_blat_results[mapped_blat_results['short_header'] == header_of_interest]['sequence_without_adapter'])

header
GC_Selvarajan            165
C_positive_heart_CAD      48
GC_Mendelian_variants     46
GC_Atrial_fib             21
GC_Mohlke                 16
GC_Liang                   8
GC_Kircher                 5
Name: count, dtype: int64

In [24]:
# all regions for reference sequences
mapped_blat_results
# match reference to alternative with variant map table and add the same region to the alternative sequence (get sequence context for the alternative sequence as well)

# modify header "~" to "," because of the variant map table
# mapped_blat_results['match_header'] = mapped_blat_results['header'].str.replace("~", ",")
# mapped_blat_results['match_header'] = mapped_blat_results['match_header'].str.replace("*", ">")
mapped_blat_results['match_header'] = mapped_blat_results['header'].apply(control_ID_processing_variant_mapping)
# left join c_variant_table_df to mapped_blat_results on short_header and REF_ID
merged_blat_variant_result = mapped_blat_results.merge(c_variant_table_df, left_on='match_header', right_on='REF_ID', how='left')
merged_blat_variant_result['Variant'].isna().value_counts()




Variant
False    649
True       5
Name: count, dtype: int64

In [25]:
# investiage not matchable sequences
not_in_variant_table = merged_blat_variant_result[merged_blat_variant_result['Variant'].isna()]
# not_in_variant_table['short_header'].str.split(':').str[0].value_counts()
not_in_variant_table['match_header'].to_list()

['GC_Selvarajan:REF_rs216222|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Selvarajan:REF_rs754064|STARR-seq-HepG2_fwd_tile1-1',
 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls,NC000001.11|230159168|C|T|MohlkeHepControls,NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3_headerDuplicate2_2_headerDuplicate1_2',
 'GC_Mendelian_variants:REF_chr8:11703860G>T|GATA4_headerDuplicate2_2_headerDuplicate1_2',
 'GC_Mendelian_variants:REF_chr8:11703890AG>A|GATA4_headerDuplicate2_2_headerDuplicate1_2']

In [26]:
# investigate with sequences which could be matched and go on with the process
succ_merged_blat_variant_result = merged_blat_variant_result[~merged_blat_variant_result['Variant'].isna()]
# succ_merged_blat_variant_result['header'].nunique() # 304
succ_merged_blat_variant_result
## multiple alternatives per region
succ_merged_blat_variant_result['ALT_ID'].nunique() # 649 => ALT_ID is unique

649

In [27]:
succ_merged_blat_variant_result # is the table for all variants from controls
# for each

,header,sequence,sequence_without_adapter,short_header,sequence_length,Q_name,strand,chrom,chromStart,chromEnd,match_header,Variant,Region,REF_ID,ALT_ID
0,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,AGGACCGGATCAACTCAGCTGCCCATGCTGGGACTGTGATTTTTTG...,CAGCTGCCCATGCTGGGACTGTGATTTTTTGTATCCTGAGTTACAC...,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,300,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,+,chr1,154813248,154813518,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib:rs74541936,GC_Atrial_fib:rs74541936|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs74541936|KCNN3|STARR-seq-A...
1,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGAGCCCTTCTGGGGGCCCTGGCCACTGGCC...,AGAGCCCTTCTGGGGGCCCTGGCCACTGGCCACTGGTGGAAGTGTT...,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,300,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,+,chr1,154839744,154840014,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib:rs34292822,GC_Atrial_fib:rs34292822|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs34292822|KCNN3|STARR-seq-A...
2,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGGAAAGGCACTGGAAATTGTACTTACTCCA...,AGGAAAGGCACTGGAAATTGTACTTACTCCATTTGGTTTGTCTATT...,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,300,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,+,chr1,154840018,154840288,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib:rs12754189,GC_Atrial_fib:rs12754189|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs12754189|KCNN3|STARR-seq-A...
3,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,AGGACCGGATCAACTTTTGCAAAGGTATGGTTGGTGGATGGAGAAA...,TTTGCAAAGGTATGGTTGGTGGATGGAGAAAAAGCGGCGTGTGAGA...,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,300,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,+,chr1,154840331,154840601,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib:rs36088503,GC_Atrial_fib:rs36088503|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs36088503|KCNN3|STARR-seq-A...
4,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,AGGACCGGATCAACTACAAATTGCTAACTGAGTGTAGAATAACAGG...,ACAAATTGCTAACTGAGTGTAGAATAACAGGGCCCCATGGGGTAAG...,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,300,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,+,chr1,154860467,154860737,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib:rs76749863,"GC_Atrial_fib:rs1218584|KCNN3|STARR-seq-AF,rs7...",GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
649,C_positive_heart_CAD:REF_rs7865618,AGGACCGGATCAACTTATTGATAACAGGGGATGGATTCTTGTGGAC...,TATTGATAACAGGGGATGGATTCTTGTGGACAAAAAAATTTAGAAT...,C_positive_heart_CAD:REF_rs7865618,300,C_positive_heart_CAD:REF_rs7865618,+,chr9,22030870,22031140,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618
650,C_positive_heart_CAD:REF_rs4977757,AGGACCGGATCAACTCTGATGGGCTTCCCTTTGTGGGTAACCCGAC...,CTGATGGGCTTCCCTTTGTGGGTAACCCGACCTTTCTCTCTGGCTG...,C_positive_heart_CAD:REF_rs4977757,300,C_positive_heart_CAD:REF_rs4977757,+,chr9,22094195,22094465,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757
651,C_positive_heart_CAD:REF_rs1537373,AGGACCGGATCAACTAGAAAACCATACCCACTTTCCCACATATCCC...,AGAAAACCATACCCACTTTCCCACATATCCCAACTATGACTGGGCA...,C_positive_heart_CAD:REF_rs1537373,300,C_positive_heart_CAD:REF_rs1537373,+,chr9,22103206,22103476,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537

In [81]:
# investigate which sequences could not be found
set(designd_sequences_controls['short_header']) - set(perfect_matches['Q_name'])
# for these headers no perfect match was found according to blat: 
# {'GC_Mendelian_variants:REF_chr8:11703860G*T|GATA4',
#  'GC_Mendelian_variants:REF_chr8:11703890AG*A|GATA4',
#  'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3'}

{'GC_Mendelian_variants:REF_chr8:11703860G*T|GATA4',
 'GC_Mendelian_variants:REF_chr8:11703890AG*A|GATA4',
 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3'}

In [85]:
# blat_results[blat_results['Q_name'] == 'GC_Mendelian_variants:REF_chr8:11703860G*T|GATA4'] # has one gap
# blat_results[blat_results['Q_name'] == 'GC_Mohlke:REF_NC000001.11|230158967|C|A|MohlkeHepControls~NC000001.11|230159168|C|T|MohlkeHepControls~NC000001.11|230159329|CTTAAAGTGTTCAGCACTCCCCT|CT|MohlkeHepControls_fwd_tile3-3'] # has one gap



,match,mismatch,rep_match,Ns,Q_gap_count,Q_gap_bases,T_gap_count,T_gap_bases,strand,Q_name,...,Q_start,Q_end,T_name,T_size,T_start,T_end,block_count,blockSizes,qStarts,tStarts
5151,270,0,0,0,0,0,1,21,+,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,...,0,270,NC_000001.11,248956422,230159136,230159427,2,"193,77,","0,193,","230159136,230159350,"


##### check length of the header
- found out that GC_Kircher header are way too long => shorten them with hash value of sequence

In [33]:
designd_sequences_controls['header_length'] = designd_sequences_controls['header'].apply(len)
pd.DataFrame(designd_sequences_controls['header_length'].value_counts())

designd_sequences_controls[designd_sequences_controls['header_length'] > 150]['header'].apply(hf.get_label).value_counts()

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1481878/2629499642.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  designd_sequences_controls['header_length'] = designd_sequences_controls['header'].apply(len)


header
GC_Kircher              203
C_SLEA                  156
GC_Mohlke                 5
C_negative_neuron_NP      2
Name: count, dtype: int64

##### Change headers of GC_Kircher

In [41]:
# shorten GC_Kircher headers with hashlib
# get kircher control sequences in design file
kircher_design_sequences = designd_sequences[designd_sequences['header'].str.contains('GC_Kircher')]
kircher_design_sequences['short_header'] = kircher_design_sequences.apply(lambda x: shorten_GC_Kircher_header(x['header'], x['sequence']), axis=1)
kircher_design_sequences
# remove adapter sequences
kircher_design_sequences['sequence_without_adapter'] = kircher_design_sequences['sequence'].str.slice(15, 285)

/data/gpfs-1/users/kisa11_c/scratch/tmp/hpc-cpu-1/ipykernel_1481878/2124048503.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  kircher_design_sequences['short_header'] = kircher_design_sequences.apply(lambda x: shorten_GC_Kircher_header(x['header'], x['sequence']), axis=1)


,header,sequence,short_header
74399,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTAAAGCCCTGTCCGGTGAGGGGGCAGAAGGAC...,GC_Kircher:oligo_04fb6df8b494f179663bb0f96df02330
74400,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTAGCCCCTCTCCTTTTCCTGGACTCTGGCCGT...,GC_Kircher:oligo_c90ef99e4413a4641966516092135115
74401,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTCAGTCATGTGTTAAGTTGCGCTTCTTTGCTG...,GC_Kircher:oligo_29128b30a034134e355f2395660ed4bf
74402,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTTCTGGGTTCTGGTGTCCACTCACCCACCCCA...,GC_Kircher:oligo_f0ee3d694655151f8927bcc8c8c58d8d
74403,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTATTGGCTCTCTTCTTCAAAGGACCAGGTCCT...,GC_Kircher:oligo_9097dad4e5129cf538e329d5e2643ee7
...,...,...,...
74597,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTTCTGGGTTCTGGTGTCCACTCACCCACCCCA...,GC_Kircher:oligo_6984075ab84fb56418241d5108c297f0
74598,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTATTGGCTCTCTTCTTCAAAGGACCAGGTCCT...,GC_Kircher:oligo_76f7c1f7be445b10ca05e2305f1a21a9
74599,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTTCTGGGTTCTGGTGTCCACTCACCCACCCCA...,GC_Kircher:oligo_eef8679d0ac227321070f45be532430d
74600,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,AGGACCGGATCAACTATTGGCTCTCTTCTTCAAAGGACCAGGTCCT...,GC_Kircher:oligo_59a21a738e3297a3e9c50fdaee6e23d6


In [36]:
# check length of the mentioned controls
# designd_sequences_controls[designd_sequences_controls['header'].apply(hf.get_label) == 'C_SLEA'] # can we leave as it is
# designd_sequences_controls[designd_sequences_controls['header'].apply(hf.get_label) == 'GC_Mohlke'] # can we leave as it is
# designd_sequences_controls[designd_sequences_controls['header'].apply(hf.get_label) == 'C_negative_neuron_NP'] # can we leave as it is

# write designd_sequences_controls as fasta
output_path = '/data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/05_variant_region_list/results/control_sequences/control_sequence.fasta'

with open(output_path, 'w') as f:
    for index, row in designd_sequences_controls.iterrows():
        f.write(f">{row['header']}\n{row['sequence_without_adapter']}\n")


,header,sequence,sequence_without_adapter,header_length
76016,C_negative_neuron_NP:GW18_PFC_ABC_chr15_894002...,AGGACCGGATCAACTAGGCGCGATACGAACCCGTGGGAGCCTCCCC...,AGGCGCGATACGAACCCGTGGGAGCCTCCCCAACCCCGCAGTCCCA...,75
76017,C_negative_neuron_NP:NGN2_iPSC_ABC_chr4_112626...,AGGACCGGATCAACTCGAAAAGTGTGTGAAGTGTGAATTATATCTC...,CGAAAAGTGTGTGAAGTGTGAATTATATCTCAATAAAGCTGTTAAA...,76
76018,C_negative_neuron_NP:Midfetal_Cortex_Trevino_c...,AGGACCGGATCAACTGGCTGAGAGGCCTGATTCCTTCCACGCATCA...,GGCTGAGAGGCCTGATTCCTTCCACGCATCACAACCTGAAAATCGC...,85
76019,C_negative_neuron_NP:NGN2_iPSC_ABC_chr2_275913...,AGGACCGGATCAACTCATCTGTACTGACGACGGAATCCCAGCAGGA...,CATCTGTACTGACGACGGAATCCCAGCAGGACACCTTCACTCACTT...,72
76020,C_negative_neuron_NP:Fetal_Cerebrum_Cicero_chr...,AGGACCGGATCAACTTCTCCGCCCAGGGCAGCAGCGCGCGGGGCCC...,TCTCCGCCCAGGGCAGCAGCGCGCGGGGCCCCCGGGAGCCGAAGAG...,83
...,...,...,...,...
76228,C_negative_neuron_NP:NGN2_iPSC_ABC_chr3_427240...,AGGACCGGATCAACTCTTCACATCCTCACATCAACCAGCCGTTGGA...,CTTCACATCCTCACATCAACCAGCCGTTGGATACAGGCTGTCCATG...,72
76229,C_negative_neuron_NP:GW18_PFC_ABC_chr11_133690...,AGGACCGGATCAACTGGCCCTCAGGTGTCTCGAGATGGCCCGGGCT...,GGCCCTCAGGTGTCTCGAGATGGCCCGGGCTCCGAGCGCTCCCGGC...,72
76230,C_negative_neuron_NP:NGN2_iPSC_ABC_chr15_32862...,AGGACCGGATCAACTGGAGACCTGAAGGCTAATTAATGATGACACC...,GGAGACCTGAAGGCTAATTAATGATGACACCGGAGACTGGCAGTGC...,75
76231,C_negative_neuron_NP:GW18_PFC_ABC_Midfetal_Cor...,AGGACCGGATCAACTCCCCCACCATGCCCTCCTTTTCCTAACTGGG...,CCCCCACCATGCCCTCCTTTTCCTAACTGGGCTGAGGGAGGCTGTT...,99


In [ ]:
references = designd_sequences_controls[designd_sequences_controls['header'].str.contains('ALT')]
references = designd_sequences_controls[designd_sequences_controls['header'].str.contains('')]
references

,header,sequence,header_without_adapter
73962,GC_Atrial_fib:ALT_rs74541936|KCNN3|STARR-seq-A...,AGGACCGGATCAACTCAGCTGCCCATGCTGGGACTGTGATTTTTTG...,CAGCTGCCCATGCTGGGACTGTGATTTTTTGTATCCTGAGTTACAC...
73963,GC_Atrial_fib:ALT_rs34292822|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGAGCCCTTCTGGGGGCCCTGGCCACTGGCC...,AGAGCCCTTCTGGGGGCCCTGGCCACTGGCCACTGGTGGAAGTGTT...
73964,GC_Atrial_fib:ALT_rs12754189|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGGAAAGGCACTGGAAATTGTACTTACTCCA...,AGGAAAGGCACTGGAAATTGTACTTACTCCATTTGGTTTGTCTATT...
73965,GC_Atrial_fib:ALT_rs36088503|KCNN3|STARR-seq-A...,AGGACCGGATCAACTTTTGCAAAGGTATGGTTGGTGGATGGAGAAA...,TTTGCAAAGGTATGGTTGGTGGATGGAGAAAAAGCGGCGTGTGAGA...
73966,GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF...,AGGACCGGATCAACTACAAATTGCTAACTGAGTGTAGAATAACAGG...,ACAAATTGCTAACTGAGTGTAGAATAACAGGGCCCCATGGGGTAAG...
...,...,...,...
74904,C_positive_heart_CAD:ALT_rs4977757_rs4977757,AGGACCGGATCAACTCTGATGGGCTTCCCTTTGTGGGTAACCCGAC...,CTGATGGGCTTCCCTTTGTGGGTAACCCGACCTTTCTCTCTGGCTG...
74905,C_positive_heart_CAD:ALT_rs1537373_rs1537373,AGGACCGGATCAACTAGAAAACCATACCCACTTTCCCACATATCCC...,AGAAAACCATACCCACTTTCCCACATATCCCAACTATGACTGGGCA...
74906,C_positive_heart_CAD:ALT_rs10811656_rs10811656,AGGACCGGATCAACTAAATTAAAAGCTTCTAAACTAACAAACAGCC...,AAATTAAAAGCTTCTAAACTAACAAACAGCCAATTTGTGGAGTGTC...
74907,C_positive_heart_CAD:ALT_rs507666_rs507666,AGGACCGGATCAACTATTAAGACAAAAAAGGGAAAACAAAGACCAC...,ATTAAGACAAAAAAGGGAAAACAAAGACCACAAAGGAGGGACAGGG...


In [ ]:
def set_modified_chromosome(row):
    """i.e. from NC_000001.11 to chr1, ..."""
    if '23' in row['T_name']:
        return 'chrX'
    if '24' in row['T_name']:
        return 'chrY'
    chr_number = int(row['T_name'].split('_')[1].split('.')[0])
    return 'chr%s'%(chr_number) 

In [ ]:
# create tsv from blat results
! tail -n +6 /data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_control_sequences_matched_file.psl > /data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_control_sequence_result_without_header.tsv

In [ ]:
# load blat results and check if only one perfect match of each sequence

header_names = ['match', 'mismatch', 'rep_match', 'Ns', 'Q_gap_count', 'Q_gap_bases', 'T_gap_count', 'T_gap_bases', 'strand', 'Q_name', 'Q_size', 'Q_start', 'Q_end', 'T_name', 'T_size', 'T_start', 'T_end', 'block_count', 'blockSizes', 'qStarts', 'tStarts']
path_to_blat_results = '/data/cephfs-2/unmirrored/groups/ag-kircher/MPRA/kilian_projects/hg38/ncbi_dataset/data/GCF_000001405.26/blat_control_sequence_result_without_header.tsv'
blat_results = pd.read_csv(path_to_blat_results, sep="\t", header=None)
blat_results.columns = header_names
blat_results

# filter: number of matches=270, 
perfect_matches = blat_results[blat_results['match'] == 270]
perfect_matches.shape # 3635
# filter for blockSizes = 270
perfect_matches = perfect_matches[perfect_matches['blockSizes'] == '270,']
perfect_matches.shape # 3626
# check all columns for unique values and did not find any suspecious values
perfect_matches.T_name.value_counts() # 0 
# filter for matches startwith "NC_" in T_name
perfect_matches = perfect_matches[perfect_matches['T_name'].str.startswith('NC_')]
perfect_matches.shape # 3528
perfect_matches.Q_name.nunique() # 3518

# get subset of interesting columns
interesting_blat_results = ['Q_name', 'match', 'strand', 'T_name', 'T_start', 'T_end']
interesting_blat_results

perfect_matches = perfect_matches[interesting_blat_results]
perfect_matches

# GC_Kircher are too long for Blat again: shorten them



,Q_name,match,strand,T_name,T_start,T_end
0,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,270,+,NC_000007.14,116516656,116516926
1,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154813248,154813518
2,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154839744,154840014
3,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154840018,154840288
4,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,270,+,NC_000001.11,154840331,154840601
...,...,...,...,...,...,...
37979,MK:tile_18181|chr17-58659383+58659652|reference,270,+,NC_000017.11,58659382,58659652
37980,MK:tile_37243|chr6-97306567+97306836|reference,270,+,NC_000006.12,97306566,97306836
37981,MK:newcore_229767|chr20-22686062+22686331|refe...,270,+,NC_000020.11,22686061,22686331
37983,MK:ZNF493|chr19-21396827+21397096|reference,270,+,NC_000019.10,21396826,21397096


In [ ]:
c_variant_table_df.head()

,Variant,Region,label,REF_ID,ALT_ID
0,GC_Atrial_fib:rs74541936,GC_Atrial_fib:rs74541936|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs74541936|KCNN3|STARR-seq-A...
1,GC_Atrial_fib:rs34292822,GC_Atrial_fib:rs34292822|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs34292822|KCNN3|STARR-seq-A...
2,GC_Atrial_fib:rs12754189,GC_Atrial_fib:rs12754189|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs12754189|KCNN3|STARR-seq-A...
3,GC_Atrial_fib:rs36088503,GC_Atrial_fib:rs36088503|KCNN3|STARR-seq-AF_fw...,GC_Atrial_fib,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,GC_Atrial_fib:ALT_rs36088503|KCNN3|STARR-seq-A...
4,GC_Atrial_fib:rs76749863,"GC_Atrial_fib:rs1218584|KCNN3|STARR-seq-AF,rs7...",GC_Atrial_fib,GC_Atrial_fib:REF_rs1218584|KCNN3|STARR-seq-AF...,GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF...


In [ ]:
c_variant_table_df[c_variant_table_df['Variant'].str.contains('GC_Kircher')]

,Variant,Region,label,REF_ID,ALT_ID
249,GC_Kircher:NC000001_11_109274794_C_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...
250,GC_Kircher:NC000001_11_109274836_C_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...
251,GC_Kircher:NC000001_11_109274836_C_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...
252,GC_Kircher:NC000001_11_109274840_A_C,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...
253,GC_Kircher:NC000001_11_109274840_A_C,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...
...,...,...,...,...,...
442,GC_Kircher:NC000001_11_109275171_G_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...
443,GC_Kircher:NC000001_11_109275171_G_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...
444,GC_Kircher:NC000001_11_109275179_A_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...
445,GC_Kircher:NC000001_11_109275179_A_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...


In [11]:
header_length = designd_sequences['header'].apply(len)

NameError: name 'designd_sequences' is not defined

In [183]:
# shorten GC_Kircher headers with vcf 
kircher_vcf = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/final_design/GC_Kircher/variants.vcf.gz'
kircher_bed = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/final_design/GC_Kircher/regions.bed.gz'
kircher_bed_file = pd.read_csv(kircher_bed, sep="\t", header=None)
kircher_bed_file.columns = ['chr', 'start', 'end', 'ID', 'score', 'strand']
kircher_bed_file
kircher_bed_file['ID'] = kircher_bed_file['ID'].str.replace(",", "~")
kircher_bed_file['ID'].to_list()
kircher_design_sequences = designd_sequences[designd_sequences['header'].str.contains('GC_Kircher')]
# add region id to design_sequences
all_references = kircher_design_sequences.merge(c_variant_table_df, left_on='header', right_on='REF_ID', how='left')
all_references = all_references[~all_references['Variant'].isna()] # 198 
all_alternatives = kircher_design_sequences.merge(c_variant_table_df, left_on='header', right_on='ALT_ID', how='left')
all_alternatives = all_alternatives[~all_alternatives['Variant'].isna()] # 198 

# left join with kircher_bed_file
all_references
all_alternatives

# merge all_reference and all_alternatives with vcf file
gc_kircher_vcf_path = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/design/final_design/results/final_design/GC_Kircher/variants.vcf.gz'
gc_kircher_vcf = pd.read_csv(gc_kircher_vcf_path, sep="\t", skiprows=5)
gc_kircher_vcf
all_reference_vcf = all_references.merge(gc_kircher_vcf, left_on='Variant', right_on='ID', how='left')
all_alternative_vcf = all_alternatives.merge(gc_kircher_vcf, left_on='Variant', right_on='ID', how='left')
all_reference_vcf
all_alternative_vcf

# # left join all_reference_vcf and all_alternative_vcf with kircher_design_sequences and add ID column
header_id_ref = all_reference_vcf[['header', 'ID', '#CHROM', 'POS']]
header_id_alt = all_alternative_vcf[['header', 'ID', '#CHROM', 'POS']]
kircher_design_sequences = kircher_design_sequences.merge(header_id_ref, left_on='header', right_on='header', how='left')
kircher_design_sequences = kircher_design_sequences.merge(header_id_alt, left_on='header', right_on='header', how='left')
kircher_design_sequences
# merge columns ID_x and ID_y and rename to ID then remove same for POS and #CHROM
kircher_design_sequences['ID'] = kircher_design_sequences['ID_x'].fillna(kircher_design_sequences['ID_y'])
kircher_design_sequences.drop(columns=['ID_x', 'ID_y'], inplace=True)
kircher_design_sequences['CHROM'] = kircher_design_sequences['#CHROM_x'].fillna(kircher_design_sequences['#CHROM_y'])
kircher_design_sequences.drop(columns=['#CHROM_x', '#CHROM_y'], inplace=True)
kircher_design_sequences['POS'] = kircher_design_sequences['POS_x'].fillna(kircher_design_sequences['POS_y'])
kircher_design_sequences.drop(columns=['POS_x', 'POS_y'], inplace=True)
kircher_design_sequences['header'].nunique() # 203
# kircher_design_sequences['ID'].nunique() # 203



# show duplicates
# kircher_design_sequences[kircher_design_sequences['ID'].duplicated(keep=False)]




203

In [119]:
mapped_blat_results = designd_sequences_controls.merge(perfect_matches, left_on='header', right_on='Q_name', how='left')

mapped_blat_results[~mapped_blat_results['Q_name'].isna()] # 3523 
# mapped_blat_results[mapped_blat_results['Q_name'].isna()]

,header,sequence,header_without_adapter,Q_name,match,strand,T_name,T_start,T_end
0,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,AGGACCGGATCAACTTCATTTCATTATAATCAAAAAGGATTTTTAA...,TCATTTCATTATAATCAAAAAGGATTTTTAATTACTTACTTGTTAA...,GC_Atrial_fib:rs7795510|CAV1|STARR-seq-AF~rs78...,270.0,+,NC_000007.14,116516656.0,116516926.0
1,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,AGGACCGGATCAACTCAGCTGCCCATGCTGGGACTGTGATTTTTTG...,CAGCTGCCCATGCTGGGACTGTGATTTTTTGTATCCTGAGTTACAC...,GC_Atrial_fib:REF_rs74541936|KCNN3|STARR-seq-A...,270.0,+,NC_000001.11,154813248.0,154813518.0
2,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGAGCCCTTCTGGGGGCCCTGGCCACTGGCC...,AGAGCCCTTCTGGGGGCCCTGGCCACTGGCCACTGGTGGAAGTGTT...,GC_Atrial_fib:REF_rs34292822|KCNN3|STARR-seq-A...,270.0,+,NC_000001.11,154839744.0,154840014.0
3,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,AGGACCGGATCAACTAGGAAAGGCACTGGAAATTGTACTTACTCCA...,AGGAAAGGCACTGGAAATTGTACTTACTCCATTTGGTTTGTCTATT...,GC_Atrial_fib:REF_rs12754189|KCNN3|STARR-seq-A...,270.0,+,NC_000001.11,154840018.0,154840288.0
4,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,AGGACCGGATCAACTTTTGCAAAGGTATGGTTGGTGGATGGAGAAA...,TTTGCAAAGGTATGGTTGGTGGATGGAGAAAAAGCGGCGTGTGAGA...,GC_Atrial_fib:REF_rs36088503|KCNN3|STARR-seq-A...,270.0,+,NC_000001.11,154840331.0,154840601.0
...,...,...,...,...,...,...,...,...,...
4719,MK:tile_18181|chr17-58659383+58659652|reference,AGGACCGGATCAACTTTTTTAGGCATCGCTTCGAAACTACCGCTTC...,TTTTTAGGCATCGCTTCGAAACTACCGCTTCTTCGTATGACACATC...,MK:tile_18181|chr17-58659383+58659652|reference,270.0,+,NC_000017.11,58659382.0,58659652.0
4720,MK:tile_37243|chr6-97306567+97306836|reference,AGGACCGGATCAACTTTTTTATTGTACTAATCAGTCTCAAGATGTT...,TTTTTATTGTACTAATCAGTCTCAAGATGTTCTCTGTGTTAATTGG...,MK:tile_37243|chr6-97306567+97306836|reference,270.0,+,NC_000006.12,97306566.0,97306836.0
4721,MK:newcore_229767|chr20-22686062+22686331|refe...,AGGACCGGATCAACTTTTTTCTCGTTTCATGAGTAGTTGATAATTC...,TTTTTCTCGTTTCATGAGTAGTTGATAATTCAGCAAAGGTTTCTTA...,MK:newcore_229767|chr20-22686062+22686331|refe...,270.0,+,NC_000020.11,22686061.0,22686331.0
4722,MK:ZNF493|chr19-21396827+21397096|reference,AGGACCGGATCAACTTTTTTTAAATTTGTGAATAGCCATATATTAT...,TTTTTTAAATTTGTGAATAGCCATATATTATTTATAAGTGCTTATG...,MK:ZNF493|chr19-21396827+21397096|reference,270.0,+,NC_000019.10,21396826.0,21397096.0


In [122]:
# check which header is in perfect_matches and not in mapped_blat_results
found_blat_results = set(perfect_matches['Q_name'])
matched_blat_results_header = set(mapped_blat_results['header'])

not_found_blat_results = found_blat_results - matched_blat_results_header
for header in not_found_blat_results:
    print(header)

GC_Kircher:REF_NC000001.11|109274794|C|T|KircherControls~NC000001.11|109274836|C|T|KircherControls~NC000001.11|109274840|A|C|KircherControls~NC000001.11|109274845|C|A|KircherControls~NC000001.11|109274846|T|G|KircherControls~NC000001.11|109274852|T|G|KircherControls~NC000001.11|109274857|T|C|KircherControls~NC000001.11|109274860|C|G|KircherControls~NC000001.11|109274865|C|A|KircherControls~NC000001.11|109274869|G|C|KircherControls~NC000001.11|109274884|G|C|KircherControls~NC000001.11|109274885|T|C|KircherC


#### combine control variant region list and and cardiac_neuro_cava_random variants

In [87]:
# # # replace "," with "~" for the controls because current design file has only ~
# # rename REF_ID to REF_ID_with_tilde and ALT_ID to ALT_ID_with_tilde
# c_variant_table_df.rename(columns={"REF_ID": "REF_ID_with_comma", "ALT_ID": "ALT_ID_with_comma"}, inplace=True)
# c_variant_table_df['REF_ID'] = c_variant_table_df['REF_ID_with_comma'].str.replace(",", "~")
# c_variant_table_df['ALT_ID'] = c_variant_table_df['ALT_ID_with_comma'].str.replace(",", "~")
# # remove REF_ID_with_comma and ALT_ID_with_comma
# c_variant_table_df.drop(columns=["REF_ID_with_comma", "ALT_ID_with_comma"], inplace=True)
# c_variant_table_df

c_variant_table_df['label'] = c_variant_table_df['Region'].apply(hf.get_label)

# # load design file
# designd_sequences = hf.fasta_to_dataframe(config['files']['final_design']['design_fasta'])


# headers = designd_sequences['header'].str.lower().to_list()
# ref_ids = c_variant_table_df['REF_ID'].str.lower().to_list()
# alt_ids = c_variant_table_df['ALT_ID'].str.lower().to_list()

# def check_if_in_ref_alt_ids(header):
#     """Check if all ref_ids and alt_ids in the header column"""
#     if header.lower() in ref_ids:
#         return True
#     elif header.lower() in alt_ids:
#         return True
#     else:
#         return False

# def check_if_ref_alt_id_in_header(id):
#     if id.lower() in headers:
#         return True
#     else:
#         return False

# # check if all ref_ids and alt_ids in the header column
# designd_sequences_controls = designd_sequences[designd_sequences['header'].apply(hf.is_control)]
# designd_sequences_controls # 6275 control sequences

# # check if all ref_ids and alt_ids in the header column
# designd_sequences_controls[~designd_sequences_controls['header'].apply(check_if_in_ref_alt_ids)]['header'].to_list()

c_variant_table_df[~c_variant_table_df['REF_ID'].str.lower().isin(headers)]['label'].value_counts()
# label
# GC_Mendelian_variants    174
# GC_Mohlke                  2
# C_positive_heart_CAD       1
c_variant_table_df[~c_variant_table_df['REF_ID'].str.lower().isin(headers)]
# found ids of Mendelian: > => *
c_variant_table_df[~c_variant_table_df['REF_ID'].str.lower().isin(headers)].shape #(14, 5)
c_variant_table_df[~c_variant_table_df['REF_ID'].str.lower().isin(headers)]
# c_variant_table_df[~c_variant_table_df['ALT_ID'].str.lower().isin(headers)].shape #(17, 5)
# c_variant_table_df[~c_variant_table_df['ALT_ID'].str.lower().isin(headers)]



# c_variant_table_df['ALT_ID'].str.lower().isin(headers).value_counts()

# GC_Mendelian_variants:REF_chr7:156791472C*T|SHH


,Variant,Region,label,REF_ID,ALT_ID
230,GC_Mohlke:NC000001_11_230158967_C_A,GC_Mohlke:NC000001.11|230158967|C|A|MohlkeHepC...,GC_Mohlke,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...
236,GC_Mohlke:NC000010_11_100315721_G_A,GC_Mohlke:NC000010.11|100315721|G|A|MohlkeNonH...,GC_Mohlke,GC_Mohlke:REF_NC000010.11|100315721|G|A|Mohlke...,GC_Mohlke:ALT_NC000010.11|100315721|G|A|Mohlke...
502,GC_Mendelian_variants:chr7:156791413A>C|SHH,GC_Mendelian_variants:chr7:156791472C>T|SHH,GC_Mendelian_variants,GC_Mendelian_variants:REF_chr7:156791472C*T|SHH,GC_Mendelian_variants:ALT_chr7:156791472C*T|SH...
508,GC_Mendelian_variants:chr7:156791459T>C|SHH,GC_Mendelian_variants:chr7:156791472C>T|SHH,GC_Mendelian_variants,GC_Mendelian_variants:REF_chr7:156791472C*T|SHH,GC_Mendelian_variants:ALT_chr7:156791472C*T|SH...
516,GC_Mendelian_variants:chr7:156791472C>G|SHH,GC_Mendelian_variants:chr7:156791472C>T|SHH,GC_Mendelian_variants,GC_Mendelian_variants:REF_chr7:156791472C*T|SHH,GC_Mendelian_variants:ALT_chr7:156791472C*T|SH...
527,GC_Mendelian_variants:chr7:156791472C>T|SHH,GC_Mendelian_variants:chr7:156791472C>T|SHH,GC_Mendelian_variants,GC_Mendelian_variants:REF_chr7:156791472C*T|SHH,GC_Mendelian_variants:ALT_chr7:156791472C*T|SH...
538,GC_Mendelian_variants:chr7:156791474G>A|SHH,GC_Mendelian_variants:chr7:156791472C>T|SHH,GC_Mendelian_variants,GC_Mendelian_variants:REF_chr7:156791472C*T|SHH,GC_Mendelian_variants:ALT_chr7:156791472C*T|SH...
549,GC_Mendelian_variants:chr7:156791480G>A|SHH,GC_Mendelian_variants:chr7:156791472C>T|SHH,GC_Mendelian_variants,GC_Mendelian_variants:REF_chr7:156791472C*T|SHH,GC_Mendelian_variants:ALT_chr7:156791472C*T|SH...
559,GC_Mendelian_variants:chr7:156791542A>C|SHH,GC_Mendelian_variants:chr7:156791472C>T|SHH,GC_Mendelian_variants,GC_Mendelian_variants:REF_chr7:156791472C*T|SHH,GC_Mendelian_variants:ALT_chr7:156791472C*T|SH...
569,GC_Mendelian_variants:chr7:156791547A>G|SHH,GC_Mendelian_variants:chr7:156791472C>T|SHH,GC_Mendelian_variants,GC_Mendelian_variants:REF_chr7:156791472C*T|SHH,GC_Mendelian_variants:ALT_chr7:156791472C*T|SH...


In [ ]:
'GC_Selvarajan:REF_rs216222'

In [29]:
c_variant_table_df['Region'].to_list()

['GC_Atrial_fib:rs74541936|KCNN3|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs34292822|KCNN3|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs12754189|KCNN3|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs36088503|KCNN3|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs1218584|KCNN3|STARR-seq-AF,rs76749863|KCNN3|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs1218584|KCNN3|STARR-seq-AF,rs76749863|KCNN3|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs10908445|KCNN3|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs680084|PRRX1|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs2902635|SH3PXD2A|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs6495062|HCN4|STARR-seq-AF,rs6495063|HCN4|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs6495062|HCN4|STARR-seq-AF,rs6495063|HCN4|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs9940321|ZFHX3|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs6599220|SCN5A|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs7430477|SCN10A|STARR-seq-AF_fwd_tile1-1',
 'GC_Atrial_fib:rs9790621|PITX2|STARR-seq-AF_fwd_til

In [28]:
# investigate label distribution
c_variant_table_df['REF_ID'].to_list()
c_variant_table_df['ALT_ID'].to_list()

['GC_Atrial_fib:ALT_rs74541936|KCNN3|STARR-seq-AF_fwd_tile1-1_rs74541936',
 'GC_Atrial_fib:ALT_rs34292822|KCNN3|STARR-seq-AF_fwd_tile1-1_rs34292822',
 'GC_Atrial_fib:ALT_rs12754189|KCNN3|STARR-seq-AF_fwd_tile1-1_rs12754189',
 'GC_Atrial_fib:ALT_rs36088503|KCNN3|STARR-seq-AF_fwd_tile1-1_rs36088503',
 'GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF~rs76749863|KCNN3|STARR-seq-AF_fwd_tile1-1_rs76749863',
 'GC_Atrial_fib:ALT_rs1218584|KCNN3|STARR-seq-AF~rs76749863|KCNN3|STARR-seq-AF_fwd_tile1-1_rs1218584',
 'GC_Atrial_fib:ALT_rs10908445|KCNN3|STARR-seq-AF_fwd_tile1-1_rs10908445',
 'GC_Atrial_fib:ALT_rs680084|PRRX1|STARR-seq-AF_fwd_tile1-1_rs680084',
 'GC_Atrial_fib:ALT_rs2902635|SH3PXD2A|STARR-seq-AF_fwd_tile1-1_rs2902635',
 'GC_Atrial_fib:ALT_rs6495062|HCN4|STARR-seq-AF~rs6495063|HCN4|STARR-seq-AF_fwd_tile1-1_rs6495062',
 'GC_Atrial_fib:ALT_rs6495062|HCN4|STARR-seq-AF~rs6495063|HCN4|STARR-seq-AF_fwd_tile1-1_rs6495063',
 'GC_Atrial_fib:ALT_rs9940321|ZFHX3|STARR-seq-AF_fwd_tile1-1_rs9940321'

In [12]:
# combine c_variant_table_df and deduplicated_variant_table_df
variant_table_df = pd.concat([deduplicated_variant_table_df, c_variant_table_df]) # 47044 variants
variant_table_df

# 46374 + 670 = 47044 => passt

,Variant,Region,REF_ID,ALT_ID
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:SKI|ENSG00000157933....,cardiac_neuro_cava_random:REF_SKI|ENSG00000157...,cardiac_neuro_cava_random:ALT_SKI|ENSG00000157...
...,...,...,...,...
665,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:rs7865618,C_positive_heart_CAD:REF_rs7865618,C_positive_heart_CAD:ALT_rs7865618_rs7865618
666,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:rs4977757,C_positive_heart_CAD:REF_rs4977757,C_positive_heart_CAD:ALT_rs4977757_rs4977757
667,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:rs1537373,C_positive_heart_CAD:REF_rs1537373,C_positive_heart_CAD:ALT_rs1537373_rs1537373
668,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:rs10811656,C_positive_heart_CAD:REF_rs10811656,C_positive_heart_CAD:ALT_rs10811656_rs10811656


In [13]:
# add label
variant_table_df['label'] = variant_table_df['Variant'].apply(hf.get_label)
cardiac_neuro_cava_random_variants = variant_table_df[variant_table_df['label'] == 'cardiac_neuro_cava_random'] # 46374 

In [14]:
# cardiac_neuro_cava_random_variants[cardiac_neuro_cava_random_variants['REF_ID'].str.contains("~")] # 778  
cardiac_neuro_cava_random_variants[cardiac_neuro_cava_random_variants['ALT_ID'].str.contains("~")] # 1374

,Variant,Region,REF_ID,ALT_ID,label
3198,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:REF_CSDE1|ENSG000000...,cardiac_neuro_cava_random:ALT_CSDE1|ENSG000000...,cardiac_neuro_cava_random
3199,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:REF_CSDE1|ENSG000000...,cardiac_neuro_cava_random:ALT_CSDE1|ENSG000000...,cardiac_neuro_cava_random
3200,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:REF_CSDE1|ENSG000000...,cardiac_neuro_cava_random:ALT_CSDE1|ENSG000000...,cardiac_neuro_cava_random
3201,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:REF_CSDE1|ENSG000000...,cardiac_neuro_cava_random:ALT_CSDE1|ENSG000000...,cardiac_neuro_cava_random
3202,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:REF_CSDE1|ENSG000000...,cardiac_neuro_cava_random:ALT_CSDE1|ENSG000000...,cardiac_neuro_cava_random
...,...,...,...,...,...
46298,cardiac_neuro_cava_random:EMD|ENSG00000102119....,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,cardiac_neuro_cava_random:REF_FLNA|ENSG0000019...,cardiac_neuro_cava_random:ALT_FLNA|ENSG0000019...,cardiac_neuro_cava_random
46299,cardiac_neuro_cava_random:EMD|ENSG00000102119....,cardiac_neuro_cava_random:EMD|ENSG00000102119....,cardiac_neuro_cava_random:REF_EMD|ENSG00000102...,cardiac_neuro_cava_random:ALT_EMD|ENSG00000102...,cardiac_neuro_cava_random
46300,cardiac_neuro_cava_random:EMD|ENSG00000102119....,cardiac_neuro_cava_random:FLNA|ENSG00000196924...,cardiac_neuro_cava_random:REF_FLNA|ENSG0000019...,cardiac_neuro_cava_random:ALT_FLNA|ENSG0000019...,cardiac_neuro_cava_random
46301,cardiac_neuro_cava_random:EMD|ENSG00000102119....,cardiac_neuro_cava_random:EMD|ENSG00000102119....,cardiac_neuro_cava_random:REF_EMD|ENSG00000102...,cardiac_neuro_cava_random:ALT_EMD|ENSG00000102...,cardiac_neuro_cava_random


In [19]:
variant_table_df[variant_table_df['REF_ID'].str.contains("~")] # 778 after replacing controls "," 1067 1067 + 1663 = 2730
variant_table_df[variant_table_df['ALT_ID'].str.contains("~")] # 1374 after replacing controls "," 1663

# variant_table_df[variant_table_df['REF_ID'].str.contains(",")] # 289  
# variant_table_df[variant_table_df['ALT_ID'].str.contains(",")] # 289 

,Variant,Region,REF_ID,ALT_ID,label
3198,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:REF_CSDE1|ENSG000000...,cardiac_neuro_cava_random:ALT_CSDE1|ENSG000000...,cardiac_neuro_cava_random
3199,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:REF_CSDE1|ENSG000000...,cardiac_neuro_cava_random:ALT_CSDE1|ENSG000000...,cardiac_neuro_cava_random
3200,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:REF_CSDE1|ENSG000000...,cardiac_neuro_cava_random:ALT_CSDE1|ENSG000000...,cardiac_neuro_cava_random
3201,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:REF_CSDE1|ENSG000000...,cardiac_neuro_cava_random:ALT_CSDE1|ENSG000000...,cardiac_neuro_cava_random
3202,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,cardiac_neuro_cava_random:REF_CSDE1|ENSG000000...,cardiac_neuro_cava_random:ALT_CSDE1|ENSG000000...,cardiac_neuro_cava_random
...,...,...,...,...,...
442,GC_Kircher:NC000001_11_109275171_G_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher
443,GC_Kircher:NC000001_11_109275171_G_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher
444,GC_Kircher:NC000001_11_109275179_A_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher
445,GC_Kircher:NC000001_11_109275179_A_T,GC_Kircher:NC000001.11|109274794|C|T|KircherCo...,GC_Kircher:REF_NC000001.11|109274794|C|T|Kirch...,GC_Kircher:ALT_NC000001.11|109274794|C|T|Kirch...,GC_Kircher


#### get the sequences from the design file (deduplicated)
- first for the REF ids
- then for the ALT ids
- check them
- add the variant position to the file
- 
- combine them
-

#### get all variants and refereces from the design file and add there variant position
- variant_table_df

In [39]:
designd_sequences = hf.fasta_to_dataframe(config['files']['final_design']['design_fasta'])
# add label
designd_sequences['label'] = designd_sequences['header'].apply(hf.get_label)

In [40]:
# get all ref ids
all_ref_ids = variant_table_df['REF_ID'].to_list()
# get all alt ids
variant_table_df['ALT_ID'].to_list()

,header,sequence,label
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random
...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK


In [83]:
# check if ref ids and alt ids 
variant_table_df['REF_ID'].str.contains("~").sum() # 778
variant_table_df['REF_ID'].str.contains(",").sum() # 289
variant_table_df['ALT_ID'].str.contains(",").sum() # 289
variant_table_df['ALT_ID'].str.contains("~").sum() # 1374
# variant_table_df['Variant'].str.contains(",").sum() # 1663

1374

In [48]:
1663 + 1067 # 2730

2730

#### Investigate headers of design fasta: '~' and ',' problem
- /data/gpfs-1/groups/ag_kircher/work/MPRA/IGVF_Y1_design/design/final_design/results/final_design/design.fa.gz

In [35]:
# /data/gpfs-1/users/kisa11_c/work/coding/80K_analysis/00_helpful_functions/helpful_functions.py

designd_sequences = hf.fasta_to_dataframe(config['files']['final_design']['design_fasta'])
# add label
designd_sequences['label'] = designd_sequences['header'].apply(hf.get_label)

In [37]:
designd_sequences
# check if "~" is in header
designd_sequences[designd_sequences['header'].str.contains("~")] # number of rows: 2089 

,header,sequence,label
708,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTAAGGAAGGGAGGGAGGGAGGGAGCGATCCCT...,cardiac_neuro_cava_random
709,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTGGACCACCTCCACCAACTGTCAGCTCACATC...,cardiac_neuro_cava_random
710,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACTGTCCTTCCTGAGGCCTCCAGCATTATTG...,cardiac_neuro_cava_random
711,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTTTACTTATTTATTTATTTTTTGAGACAGGGT...,cardiac_neuro_cava_random
712,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACCCCATAGTATGCCTCCCTCCCTCTCTCCT...,cardiac_neuro_cava_random
...,...,...,...
75057,GC_Cort_Chengyu:da|chr6:135971378-135971647|2....,AGGACCGGATCAACTGCCACTTAGTTTATGGTGCTTTGTTACAGCA...,GC_Cort_Chengyu
75061,GC_Cort_Chengyu:da|chr7:13920226-13920495|2.06...,AGGACCGGATCAACTCCCACCAGTTCCTGTTTATGTTTCTGATTTG...,GC_Cort_Chengyu
75062,GC_Cort_Chengyu:da|chr7:26684437-26684706|2.12...,AGGACCGGATCAACTTTACAATGTTAACTTTCACATAAGTATATTG...,GC_Cort_Chengyu
75063,GC_Cort_Chengyu:da|chr7:28125375-28125644|2.02...,AGGACCGGATCAACTGGGGAAAAAGTGCAGCACAAGCAGCTGAATT...,GC_Cort_Chengyu


In [38]:
# which labels have sequences with "~" in the header?
designd_sequences[designd_sequences['header'].str.contains("~")]['label'].value_counts() # 2089 rows, 2089 unique labels
# cardiac_neuro_cava_random    1717
# GC_Kircher                    203
# GC_Selvarajan                 123
# GC_Cort_Chengyu                27
# GC_Atrial_fib                  11
# GC_Mohlke                       8

label
cardiac_neuro_cava_random    1717
GC_Kircher                    203
GC_Selvarajan                 123
GC_Cort_Chengyu                27
GC_Atrial_fib                  11
GC_Mohlke                       8
Name: count, dtype: int64

In [24]:
# add header_with_comma column and replace "~" with ","
designd_sequences['header_with_comma'] = designd_sequences['header'].str.replace("~", ",")

In [25]:
# check if number of previously found "~" is equal to number of "," in header_with_comma
designd_sequences[designd_sequences['header_with_comma'].str.contains(",")] # 2089 

,header,sequence,header_with_comma
708,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTAAGGAAGGGAGGGAGGGAGGGAGCGATCCCT...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...
709,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTGGACCACCTCCACCAACTGTCAGCTCACATC...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...
710,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACTGTCCTTCCTGAGGCCTCCAGCATTATTG...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...
711,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTTTACTTATTTATTTATTTTTTGAGACAGGGT...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...
712,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...,AGGACCGGATCAACTACCCCATAGTATGCCTCCCTCCCTCTCTCCT...,cardiac_neuro_cava_random:CSDE1|ENSG0000000930...
...,...,...,...
75057,GC_Cort_Chengyu:da|chr6:135971378-135971647|2....,AGGACCGGATCAACTGCCACTTAGTTTATGGTGCTTTGTTACAGCA...,GC_Cort_Chengyu:da|chr6:135971378-135971647|2....
75061,GC_Cort_Chengyu:da|chr7:13920226-13920495|2.06...,AGGACCGGATCAACTCCCACCAGTTCCTGTTTATGTTTCTGATTTG...,GC_Cort_Chengyu:da|chr7:13920226-13920495|2.06...
75062,GC_Cort_Chengyu:da|chr7:26684437-26684706|2.12...,AGGACCGGATCAACTTTACAATGTTAACTTTCACATAAGTATATTG...,GC_Cort_Chengyu:da|chr7:26684437-26684706|2.12...
75063,GC_Cort_Chengyu:da|chr7:28125375-28125644|2.02...,AGGACCGGATCAACTGGGGAAAAAGTGCAGCACAAGCAGCTGAATT...,GC_Cort_Chengyu:da|chr7:28125375-28125644|2.02...


In [26]:
# for how many rows is header == header_with_comma?
designd_sequences[designd_sequences['header'] == designd_sequences['header_with_comma']] # 78126 
# everything perfect because 78126 + 2089 = 80215

,header,sequence,header_with_comma
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,cardiac_neuro_cava_random:SKI|ENSG00000157933....
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,cardiac_neuro_cava_random:SKI|ENSG00000157933....
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,cardiac_neuro_cava_random:SKI|ENSG00000157933....
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,cardiac_neuro_cava_random:SKI|ENSG00000157933....
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,cardiac_neuro_cava_random:SKI|ENSG00000157933....
...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,MK:tile_2240|chr1-116244322+116244591|scramble...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,MK:tile_6675|chr11-2374617+2374886|scramble_ne...
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,MK:tile_18415|chr17-71181691+71181960|scramble...
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,MK:tile_14356|chr15-67031618+67031887|scramble...
